In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import joblib

In [3]:
df = pd.read_csv('train.csv')
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,30.0,1.0,Urban,Yes
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,30.0,1.0,Rural,No
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,30.0,1.0,Urban,Yes
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,10.0,30.0,1.0,Urban,Yes
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,30.0,1.0,Urban,Yes


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [5]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.000000,564.000000
mean,5403.459283,1620.887492,140.094595,46.283333,0.842199
std,6109.041673,2926.431236,89.516210,80.186259,0.364878
min,150.000000,0.000000,9.000000,10.000000,0.000000
25%,2877.500000,0.000000,95.750000,30.000000,1.000000
50%,3812.500000,1188.500000,125.500000,30.000000,1.000000
75%,5795.000000,2297.250000,162.000000,30.000000,1.000000
max,81000.000000,41667.000000,700.000000,480.000000,1.000000


In [6]:
# Categorical columns
cat_cols = ['Gender', 'Married', 'Dependents', 'Self_Employed', 'Credit_History']

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)



In [7]:
# Numeric columns
num_cols = ['LoanAmount', 'Loan_Amount_Term']

for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

In [8]:
print(df.isnull().sum())

Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64


In [9]:
df['Loan_Status'] = df['Loan_Status'].map({'Yes': 1, 'No': 0})

In [10]:
# Drop ID column
df = df.drop('Loan_ID', axis=1)

# Separate features and target
X = df.drop('Loan_Status', axis=1)
y = df['Loan_Status']

In [11]:
X = pd.get_dummies(X, drop_first=True)

print(X.head())
print(X.shape)

   ApplicantIncome  CoapplicantIncome  LoanAmount  Loan_Amount_Term  \
0             5849                0.0       125.5              30.0   
1             4583             1508.0       128.0              30.0   
2             3000                0.0        66.0              30.0   
3             2583             2358.0        10.0              30.0   
4             6000                0.0       141.0              30.0   

   Credit_History  Gender_Male  Married_Yes  Dependents_1  Dependents_2  \
0             1.0         True        False         False         False   
1             1.0         True         True          True         False   
2             1.0         True         True         False         False   
3             1.0         True         True         False         False   
4             1.0         True        False         False         False   

   Dependents_3+  Education_Not Graduate  Self_Employed_Yes  \
0          False                   False              False

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [13]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [15]:
y_pred = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8455284552845529

Confusion Matrix:
 [[22 16]
 [ 3 82]]

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.58      0.70        38
           1       0.84      0.96      0.90        85

    accuracy                           0.85       123
   macro avg       0.86      0.77      0.80       123
weighted avg       0.85      0.85      0.84       123



In [16]:
# Save Model & Scaler
joblib.dump(model, "loan_eligibility_model.pkl")
joblib.dump(scaler, "loan_scaler.pkl")
joblib.dump(X.columns, "model_features.pkl")  # save feature order

['model_features.pkl']